# Approx Lab

The goal of this lab is to gain experience using approximation algorithms.

For the purpose of this assignment, you will be implementing the knapsack problem. Before continuing, the following cell contains any dependencies you may need for this assignment.

In [36]:
import pandas as pd

## Background Info

### Knapsack Problem
We are actually going to be looking at a specific (hopefully fun) implementation of the knapsack problem.

The traditional knapsack problem asks you to pack a set of items, with given weights and values into a "knapsack" with a maximum capacity. You cannot pack more items than the capacity of the knapsack. The goal is to select a subset of items that maximize the total value with what can fit in the knapsack. The name "knapsack" comes from the similarity to a hiker trying to pack a bag without going over a certain weight.

The knapsack problem is an NP-Hard optimization problem, which is why it is one of the classic approximation algorithms.

### Specifications for this Lab
The variation that we will be doing in this assignment is referred to as an 0/1 knapsack problem, meaning that items must be completely put in the knapsack or they must be left behind (you can't have a fractional portion of an item). We will be representing this as a simple fantasy sports league - specifically Formula 1 drivers. 

You are provided with a list of F1 drivers and constructors with given costs and points as well as a budget of $100M. You will be selecting five drivers and two constructors (teams). 

### Scoring Information
Points are gained from qualification results for drivers and constructors as well as race results for drivers and constructors. 

<!-- Qualifications
- Drivers score based on their resulting position 
- Constructor score the combined total of their two drivers in qualifying plus a modifier based on how far both their drivers get. -->
<!-- 
| Drivers    | Points |
| -------- | ------- |
| Pole Position (1st Place) | 10 |
| 2nd Place | 9 |
| 3rd Place | 8 |
| 4th Place | 7 |
| 5th Place | 6 |
| 6th Place | 5 |
| 7th Place | 4 |
| 8th Place | 3 |
| 9th Place | 2 |
| 10th Place | 1 |
| 11th-20th Place | 0 |
| No Time Set | 0 |
| Disqualification | 0 | -->


<!-- | Constructors | Points |
| ----------- | ------- |
| Neither Driver Reaches Q2 | -1 |
| One Driver Reaches Q2 | 1 |
| Both Drivers Reach Q2 | 5 |
| Both Drivers Reach Q3 | 10 | -->


<!-- Race
- Drivers score based on their resulting position, position difference from starting order, and some additional bonuses
- Constructors score a combined total of their two drivers' results (excluding the Driver of the Day bonus) ~~plus a modifier based on pitstop times~~ -->

<!-- | Drivers | Points |
| -------- | ------- |
| Positions Gained | 1/position |
| Positions Lost | -1/position |
| Overtakes Made | 1/overtake |
| Fastest Lap | 10 |
| Driver of the Day | 10 | -->

<!-- | Race Result | Points |
| -------- | ------- |
| 1st Place | 25 |
| 2nd Place | 18 |
| 3rd Place | 15 |
| 4th Place | 12 |
| 5th Place | 10 |
| 6th Place | 8 |
| 7th Place | 6 |
| 8th Place | 4 |
| 9th Place | 2 |
| 10th Place | 1 |
| 11th-20th Place | 0 |
| DNF | -20 |
| Disqualification | -20 | -->

Fortunately for you, we have access to the current point standings post-2025 China GP for drivers & constructors, which are available in `drivers.csv` & `constructors.csv`.

*Note: The costs are in millions, so 30 is actually 30 M.

### Questions (Answered in Various Parts of the Lab): 
1. What is the general setup of the problem?
2. Determining the mathematical functions of the problem
3. Implementing a greedy algorithm


## A. Assessing the Problem

The values that are provided for this lab are from the points distribution PRIOR to the Japan Grand Prix (which means we can actually compare with the race results at the submission deadline!)

Create a fantasy team https://fantasy.formula1.com/en/create-team for the upcoming Grand Prix to get a better idea of what drivers, constructors, points, and the budget are. Who is on your team & how much did you spend? Don't spend too much time on this, it really is just to get familiar with the format.

*Budget: $100 Million*

*5 Drivers, 2 Constructors*

![alt text](FantasyFormulaTeam.PNG)

What represents the "items" in the knapsack? 

*The items are the F1 drivers and constructors we can choose from.*

What represents the weights?

*The weights are the costs (in millions) of each driver or constructor.*

What represents the values?

*The values are the points scored by each driver or constructor.*

In [37]:
# Load the data from the CSV files into a format that you are comfortable working with
# This can be arrays, dictionaries, or pd DataFrames
drivers_df = pd.read_csv("drivers.csv")
constructors_df = pd.read_csv("constructors.csv")
# Print/Display your data as well to confirm that you have successfully saved it.
print("F1 Drivers Data:")
display(drivers_df)
print("F1 Constructors Data:")
display(constructors_df)

F1 Drivers Data:


,Driver,Points,Cost
0,Lando Norris,100,29.6
1,Andrea Kimi Antonelli,61,19.3
2,George Russell,60,21.4
3,Max Verstappen,59,28.6
4,Oscar Piastri,55,23.0
5,Lance Stroll,33,9.3
6,Esteban Ocon,32,8.1
7,Alexaner Albon,28,12.8
8,Nico Hulkenberg,22,7.6
9,Oliver Bearman,22,6.7


F1 Constructors Data:


,Constructor,Points,Cost
0,McLaren,172,30.6
1,Mercedes,136,23.3
2,Red Bull Racing,86,25.4
3,Haas,61,8.2
4,Williams,36,13.5
5,Racing Bulls,32,8.0
6,Kick Sauber,15,6.2
7,Ferrari,13,27.1
8,Aston Martin,5,7.3
9,Alpine,-20,8.3


# B. Integer Program

Before implementing our approx algorithm, we need to first consider the integer program. 

Given the following variables, create an integer program that maximizes the total points of drivers while remaining under the budget: 
- $x_i$ is the decision variable that determines whether driver $i$ is included in the team or not
- $p_i$ is the # of points that driver $i$ is worth
- $c_i$ is the cost of driver $i$

For now, we will focus on just the drivers and ignore the constructors. We will also temporarily reduce our budget to **$50M**

While we are not planning on implementing the full integer program, it helps us quantify parts of the problem


$\text{maximize} \quad \sum_{i}^{} x_i * p_i$  \
$\text{subject to} \quad \sum_{i}^{} x_i * c_i \le 50\\
                    \quad \sum_{i}^{} x_i = 5\\
                    \quad x_i \in \{0,1\} ∀i$


## Decision Variable & Constraints

We can make helper functions for our future algorithm based on constraints that we derive when forming the IP: 
1. Which drivers should be included in our solution? 
2. What is our maximum cost?
3. What is our maximum # of drivers? 

In [ ]:
budget = 50 # in millions
max_drivers = 5  # can only pick up to 5 drivers

# Helper function to check if a driver can be added without breaking the budget
def can_add_driver(selected_drivers, new_driver, current_cost, budget=budget):
    return (
        len(selected_drivers) < max_drivers and 
        (current_cost + new_driver["Cost"]) <= budget
    )

# C. Greedy Algorithm


There are multiple ways we can go about solving the knapsack problem. For this lab, we will be focusing on a greedy algorithm and a variant of that.

Let's start with a basic greedy algorithm:
1. Calculate the value-to-cost ratio for each driver 
2. Sort drivers in non-increasing order of value-to-cost
2. Greedily select drivers from list until either budget or max drivers constraint is hit

Use the helper methods from the previous part in your implementation.

In [39]:
def value_to_cost(items): 
    drivers_df["ValueToCost"] = drivers_df["Points"] / drivers_df["Cost"]
    return drivers_df

def sort_items(items): 
    return drivers_df.sort_values(by="ValueToCost", ascending=False)

def greedy_driver_select(drivers_df, budget=50, max_drivers=5):
    drivers_df = value_to_cost(drivers_df)
    sorted_drivers = sort_items(drivers_df)
    
    selected_drivers = []
    total_cost = 0

    for index, driver in sorted_drivers.iterrows():
        if can_add_driver(selected_drivers, driver, total_cost, budget):
            selected_drivers.append(driver)
            total_cost += driver["Cost"]

    selected_df = pd.DataFrame(selected_drivers)
    return selected_df

selected_drivers_df = greedy_driver_select(drivers_df)

print("Selected Drivers (Greedy Algorithm):")
display(selected_drivers_df)

print(f"Total Cost: {selected_drivers_df['Cost'].sum()}M")
print(f"Total Points: {selected_drivers_df['Points'].sum()}")

Selected Drivers (Greedy Algorithm):


,Driver,Points,Cost,ValueToCost
6,Esteban Ocon,32,8.1,3.950617
5,Lance Stroll,33,9.3,3.548387
0,Lando Norris,100,29.6,3.378378


Total Cost: 47.0M
Total Points: 165


Try running your basic greedy algorithm. 

You may notice a few flaws:
1. The greedy algorithm is only looking at value-to-cost ratio, which may miss out on an overall improvement in value that still fits within the budget
2. Depending on your implementation, you may not currently be meeting the "5 total drivers" constraint - this is fine


## Redux Greedy Algorithm
We are going to next implement a small adjustment to this algorithm in order to improve its performance. \
This is a 2-approximation for the knapsack problem. 

1. Calculate the value-to-cost ratio for each driver
2. Sort drivers in non-increasing order of value-to-cost
3. Greedily add items until we hit an item that is too big
4. Compare the too big item against the already existing list of items & pick the better of the two

It might help to make a separate function to handle #4. 

In [40]:
def redux_greedy_driver_select(drivers_df, budget=50, max_drivers=5):
    drivers_df = value_to_cost(drivers_df.copy())
    sorted_drivers = sort_items(drivers_df)

    selected_drivers = []
    total_cost = 0
    best_single_driver = None

    for index, driver in sorted_drivers.iterrows():
        if can_add_driver(selected_drivers, driver, total_cost, budget):
            selected_drivers.append(driver)
            total_cost += driver["Cost"]
        else:
            if best_single_driver is None or driver["Points"] > best_single_driver["Points"]:
                best_single_driver = driver

    total_greedy_points = sum(driver["Points"] for driver in selected_drivers)
    
    # Compare with the best single driver that didn't fit
    if best_single_driver is not None and best_single_driver["Cost"] <= budget:
        if best_single_driver["Points"] > total_greedy_points:
            return pd.DataFrame([best_single_driver])

    return pd.DataFrame(selected_drivers)

redux_selected_drivers_df = redux_greedy_driver_select(drivers_df)

print("Redux Greedy Algorithm - Selected Drivers:")
display(redux_selected_drivers_df)

print(f"Total Cost: {redux_selected_drivers_df['Cost'].sum()}M")
print(f"Total Points: {redux_selected_drivers_df['Points'].sum()}")

Redux Greedy Algorithm - Selected Drivers:


,Driver,Points,Cost,ValueToCost
6,Esteban Ocon,32,8.1,3.950617
5,Lance Stroll,33,9.3,3.548387
0,Lando Norris,100,29.6,3.378378


Total Cost: 47.0M
Total Points: 165


# D. Reintroducing Constructors

We've been neglecting our constructors so far and cutting our budget. Let's make them feel included in our processes again. 

## Integer Program

We should start by redefining our original IP to see what constraints we need to add. \
We've added some more variables to our IP: 
- $x_j$ is the decision variable that determines whether constructor $j$ is included in the team or not 
- $p_j$ is the # of points that constructor $j$ is worth
- $c_j$ is the cost of constructor $j$

Our budget has also been increased back to **100M**. 


$\text{maximize} \quad \sum_{i}^{}x_i * p_i + \sum_{j}^{}y_j * p_j$  \
$\text{subject to} \quad \sum_{i}^{} x_i * c_i + \sum_{j}^{} y_j * c_j\le 100\\
                    \quad \sum_{i}^{} x_i = 5\\
                    \quad \sum_{j}^{} y_j = 2\\
                    \quad x_i, y_j \in \{0,1\} ∀i$ 


## Constraints 

To redefine our greedy algorithm to include constructors, we have to also write helpers that correspond to our new constraints.

In [41]:
budget = 100  # in millions
max_drivers = 5
max_constructors = 2

def can_add_item(item, selected_items, current_cost):
    """
    Check if the item can be added without violating constraints.
    """
    if current_cost + item["Cost"] > budget:
        return False

    drivers_count = sum(1 for i in selected_items if i["Type"] == "Driver")
    constructors_count = sum(1 for i in selected_items if i["Type"] == "Constructor")

    if item["Type"] == "Driver" and drivers_count >= max_drivers:
        return False
    if item["Type"] == "Constructor" and constructors_count >= max_constructors:
        return False

    return True

def count_items(selected_items):
    """
    Returns counts of drivers and constructors in the current selection.
    """
    drivers_count = sum(1 for i in selected_items if i["Type"] == "Driver")
    constructors_count = sum(1 for i in selected_items if i["Type"] == "Constructor")
    return drivers_count, constructors_count

def get_total_cost(selected_items):
    """
    Returns the total cost of the selected team.
    """
    return sum(i["Cost"] for i in selected_items)


## Greedy Algorithm

Update the greedy algorithm to calculate for buying both constructors and drivers. \
Remember, your list of non-increasing value-to-cost ratios should be a combination of both drivers & constructors. 

In [42]:
def greedy_f1(drivers_df, constructors_df):
    drivers_df = drivers_df.copy()
    constructors_df = constructors_df.copy()
    
    drivers_df["Type"] = "Driver"
    constructors_df["Type"] = "Constructor"

    all_items = pd.concat([drivers_df, constructors_df], ignore_index=True)

    all_items["Ratio"] = all_items["Points"] / all_items["Cost"]

    sorted_items = all_items.sort_values(by="Ratio", ascending=False)

    selected_items = []
    current_cost = 0

    for _, item in sorted_items.iterrows():
        if can_add_item(item, selected_items, current_cost):
            selected_items.append(item)
            current_cost += item["Cost"]

    best_single = all_items.loc[all_items["Cost"] <= budget].sort_values(by="Points", ascending=False).iloc[0]

    greedy_points = sum(item["Points"] for item in selected_items)
    if best_single["Points"] > greedy_points:
        selected_items = [best_single]

    final_team = pd.DataFrame(selected_items)
    final_team.reset_index(drop=True, inplace=True)

    return final_team


final_team_df = greedy_f1(drivers_df, constructors_df)

print("Final Fantasy F1 Team (Drivers + Constructors):")
display(final_team_df)

print(f"Total Cost: {final_team_df['Cost'].sum()}M")
print(f"Total Points: {final_team_df['Points'].sum()}")


Final Fantasy F1 Team (Drivers + Constructors):


,Driver,Points,Cost,ValueToCost,Type,Constructor,Ratio
0,NaN,61,8.2,NaN,Constructor,Haas,7.439024
1,NaN,136,23.3,NaN,Constructor,Mercedes,5.836910
2,Esteban Ocon,32,8.1,3.950617,Driver,NaN,3.950617
3,Lance Stroll,33,9.3,3.548387,Driver,NaN,3.548387
4,Lando Norris,100,29.6,3.378378,Driver,NaN,3.378378
5,Oliver Bearman,22,6.7,3.283582,Driver,NaN,3.283582
6,Nico Hulkenberg,22,7.6,2.894737,Driver,NaN,2.894737


Total Cost: 92.8M
Total Points: 406
